In [ ]:
# Assignment 7.1 — Data Processing: Canada Immigration Dataset
# Goal: Clean the United Nations immigration data for Canada to match the
# table shown in the assignment. The final table contains Country (index), 
# Continent, Region, DevName, annual immigration values from 1980 to 2013, 
# and a Total column.

In [ ]:
# 1. Import pandas and load the worksheet
# The worksheet has introductory text above the actual table. skiprows=20 tells 
# pandas to use Excel row 21 as the header.

In [4]:
import pandas as pd
from pathlib import Path

# Read the source Excel workbook from the same directory as the notebook.
input_file = Path("Canada.xlsx")
if not input_file.exists():
    raise FileNotFoundError(
        "Put 'Canada (1).xlsx' in the same folder as this notebook before running it."
    )

# Header names are on Excel row 21 (0-based row index 20).
raw = pd.read_excel(input_file, sheet_name="Canada by Citizenship", skiprows=20)
print("Raw dataset shape:", raw.shape)
print("First five records and selected source columns:")
print(raw[["Type", "Coverage", "OdName", "AreaName", "RegName", "DevName"]].head().to_string(index=False))

Raw dataset shape: (197, 43)
First five records and selected source columns:
      Type   Coverage         OdName AreaName         RegName            DevName
Immigrants Foreigners    Afghanistan     Asia   Southern Asia Developing regions
Immigrants Foreigners        Albania   Europe Southern Europe  Developed regions
Immigrants Foreigners        Algeria   Africa Northern Africa Developing regions
Immigrants Foreigners American Samoa  Oceania       Polynesia Developing regions
Immigrants Foreigners        Andorra   Europe Southern Europe  Developed regions


In [2]:
# 2. Keep the required immigration records and columns
# The example is for Immigrants / Foreigners. Other types and coverage categories 
# should not be mixed into the result. We rename the location fields and remove 
# classification codes such as AREA, REG, and DEV.

In [5]:
# Keep the reporting category shown in the assignment example.
canada = raw.loc[
    (raw["Type"] == "Immigrants") & (raw["Coverage"] == "Foreigners")
].copy()

# The worksheet uses OdName/AreaName/RegName for the displayed names.
years = list(range(1980, 2014))
required = ["OdName", "AreaName", "RegName", "DevName"] + years
missing = [column for column in required if column not in canada.columns]
if missing:
    raise ValueError(f"Required Excel columns not found: {missing}")

canada = canada[required].rename(columns={
    "OdName": "Country",
    "AreaName": "Continent",
    "RegName": "Region"
})

# Remove completely blank/footer rows and ensure each country has a name.
canada = canada.dropna(subset=["Country"])
print("Rows kept for Immigrants / Foreigners:", len(canada))
print("Columns before adding Total:", len(canada.columns))

Rows kept for Immigrants / Foreigners: 195
Columns before adding Total: 38


In [6]:
# 3. Clean year values and calculate the total
# The UN source uses .. where values are not available. For reproducing the assignment 
# table, those entries are treated as zero before summing. This is a calculation 
# convention, not a claim that the original missing values were actually zero.

In [7]:
# Convert year fields to numeric values, treating '..' and blanks as 0.
canada[years] = (
    canada[years]
    .replace("..", pd.NA)
    .apply(pd.to_numeric, errors="coerce")
    .fillna(0)
    .astype(int)
)

# Total number of recorded immigrants for each country, 1980–2013.
canada["Total"] = canada[years].sum(axis=1)

# Make Country the row index, just like the requested output.
canada = canada.set_index("Country")
canada.index.name = "Country"

print("Cleaned dataset shape:", canada.shape)
print("First five countries, selected years, and totals:")
print(canada[["Continent", "Region", "DevName", 1980, 1981, 1982, 2011, 2012, 2013, "Total"]].head().to_string())

Cleaned dataset shape: (195, 38)
First five countries, selected years, and totals:
               Continent           Region             DevName  1980  1981  1982  2011  2012  2013  Total
Country                                                                                                 
Afghanistan         Asia    Southern Asia  Developing regions    16    39    39  2203  2635  2004  58639
Albania           Europe  Southern Europe   Developed regions     1     0     0   539   620   603  15699
Algeria           Africa  Northern Africa  Developing regions    80    67    71  4325  3774  4331  69439
American Samoa   Oceania        Polynesia  Developing regions     0     1     0     0     0     0      6
Andorra           Europe  Southern Europe   Developed regions     0     0     0     0     1     1     15


In [8]:
# 4. Display the final dataframe
# This DataFrame is the requested result: country names as the index, three descriptive 
# columns, 1980–2013, and the Total column.

In [9]:
# Show the entire first five records in Jupyter's formatted table.
display(canada.head())

,Continent,Region,DevName,1980,1981,1982,1983,1984,1985,1986,...,2005,2006,2007,2008,2009,2010,2011,2012,2013,Total
Country,,,,,,,,,,,,,,,,,,,,,
Afghanistan,Asia,Southern Asia,Developing regions,16,39,39,47,71,340,496,...,3436,3009,2652,2111,1746,1758,2203,2635,2004,58639
Albania,Europe,Southern Europe,Developed regions,1,0,0,0,0,0,1,...,1223,856,702,560,716,561,539,620,603,15699
Algeria,Africa,Northern Africa,Developing regions,80,67,71,69,63,44,69,...,3626,4807,3623,4005,5393,4752,4325,3774,4331,69439
American Samoa,Oceania,Polynesia,Developing regions,0,1,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,6
Andorra,Europe,Southern Europe,Developed regions,0,0,0,0,0,0,2,...,0,1,1,0,0,0,0,1,1,15


In [10]:
# 5. Validate sample values
# Compare the computed totals to the values in the screenshot. These checks help 
# confirm that years were included correctly and that the data was not shifted by the 
# header rows.

In [11]:
expected_totals = {
    "Afghanistan": 58639,
    "Albania": 15699,
    "Algeria": 69439,
    "American Samoa": 6,
    "Andorra": 15,
}

for country, expected in expected_totals.items():
    actual = int(canada.loc[country, "Total"])
    print(f"{country}: {actual:,} (expected {expected:,})")
    assert actual == expected, f"Mismatch for {country}: got {actual}, expected {expected}"

assert list(canada.columns[:3]) == ["Continent", "Region", "DevName"]
assert list(canada.columns[3:-1]) == years
assert canada.columns[-1] == "Total"
print("All checks passed. The table matches the expected structure and sample totals.")

Afghanistan: 58,639 (expected 58,639)
Albania: 15,699 (expected 15,699)
Algeria: 69,439 (expected 69,439)
American Samoa: 6 (expected 6)
Andorra: 15 (expected 15)
All checks passed. The table matches the expected structure and sample totals.


In [ ]:
# Conclusion
# I imported the UN Canada immigration worksheet, found the real data header, selected the required immigration records, renamed and organized the requested fields, converted annual columns to numeric data, and summed annual counts for each country. The final DataFrame follows 
# the table layout in the assignment and passes checks against the screenshot's 
# first five totals.